# Capstone — Content decline rankings that survive a better test
Hicham Halfi · AI-assisted analysis · 20 October 2026

## 1. Question and case-study framing
For a FlyRank content team with a limited editor budget, can a shallow forest retrieve decline-labeled pages better than transparent rules? Week 5 appeared promising on random rows. This study asks what survives client separation and stricter feature timing. The product is a review queue, not automated publishing.

## 2. Data
Use the public bundled 30,000-row, 44-column starter snapshot covering 32 pseudonymized clients. One row per content item; trailing 90-day totals and two 30-day comparison windows. The exact calendar snapshot date is unavailable. All rows pass impressions_90d >0. This is not a warehouse-scale analysis. The label is trend_direction == down (greater than 20% impression decline). IDs are grouping keys, never features.

## 3. Methodology
Reproduce the Week-5 five-feature forest and baseline, switch to grouped client holdout, then compare previous-30-day-only features with previous impressions as baseline on the same group holdout. Use 100 trees, depth 5 and seed 42. Five grouped folds assess sensitivity. The validation notebook contains the leakage control, missingness checks and real errors; the playbook creates out-of-client scores. Previous impressions also enter the decline denominator; active-population selection and absent calendar dates limit prospective interpretation.


In [1]:
from pathlib import Path
import json
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'work/outputs/validation_metrics.json').exists())
a=json.loads((ROOT/'work/outputs/validation_metrics.json').read_text())
p=json.loads((ROOT/'work/outputs/playbook_metrics.json').read_text())
assert a['rows']==30000 and a['clients']==32 and a['invalid_leak_control_accuracy']==1
for r in a['experiments']:
    print(r['experiment'], 'model P@20',r['model']['precision_at_20'],
          'rule P@20',r['baseline']['precision_at_20'],'prevalence',r['model']['base_rate'])
print('OOF queue P@20:',p['oof_precision_at_20'],'prevalence:',p['base_rate'])


week5_random_reproduction model P@20 0.9 rule P@20 0.65 prevalence 0.5446666666666666
same_features_client_holdout model P@20 0.5 rule P@20 0.35 prevalence 0.5109524582184002
prior_window_client_holdout model P@20 0.8 rule P@20 0.6 prevalence 0.5109524582184002
OOF queue P@20: 0.75 prevalence: 0.5420666666666667


## 4. Results versus baseline
The original five-feature model measures 90% P@20 on random rows and 50% on held-out clients; its rule measures 65% and 35%, respectively. The prior-window alternative measures 80% vs 60% on the grouped holdout, where prevalence is 51.1%. Its five fold scores range from 65% to 100%; the baseline wins one fold. Global OOF top-20 precision is 75%, versus overall prevalence 54.2%. Global scores from distinct fold models are not assumed calibrated.

![Measured comparison](../figures/validation_comparison.png)

## 5. Limitations
Snapshot data, outcome-containing active-page selection, label denominator dependence, client-mix variability, small top-k samples and no independent forward-time test. No causal refresh effect, traffic gain, calibrated risk or Google-algorithm claim follows. False positives have unknown causes, not proven seasonality.

## 6. Ranked recommendations
Use the local queue to plan human review. Check measurement first; inspect stale mature assets; inspect snippet/intent for page-one low-CTR pages; inspect relevance for positions 11–20. Thresholds and time estimates are assumptions. Human approval precedes any publishing, deletion, consolidation or factual rewrite. Monitoring triggers and proposed cost scenarios are detailed in the action playbook.

![Proposed review mix](../figures/action_mix.png)

## 7. Artifacts the paper embeds
The figures and metrics above come from executed ML-09 and ML-10 notebooks. The code below rebuilds the public page from those receipts without copying any row-level queue into the publishing folder.


In [2]:
import runpy
runpy.run_path(str(ROOT/'work/scripts/build_paper.py'),run_name='__main__')
assert (ROOT/'docs/index.html').exists()
for f in ['validation_comparison.png','action_mix.png']:
    assert (ROOT/'docs/assets'/f).exists()
print('Paper source and figures are present; live deployment is verified separately.')


Built docs/index.html from real metrics
Paper source and figures are present; live deployment is verified separately.


## 8. Reproducibility and credit
See [reproduction instructions](../REPRODUCE.md), pinned audit requirements, validation_metrics.json and playbook_metrics.json. Run ML-09, ML-10, then this notebook from the repository checkout. Built on the [FlyRank ML Internship dataset](https://flyrank.ai). The source research paper in docs is discussed constructively in the validation audit. Analysis and drafting used AI assistance.

## ML-12 — Five-minute demo outline
- **0:00–0:40 / Question:** a content editor has ten review-hours; which pages should get attention?
- **0:40–1:30 / Data and method:** show starter size, 32 clients, decline label, excluded label columns, and client separation.
- **1:30–2:40 / One chart:** open validation_comparison.png. Explain 90% to 50% for the original feature set; distinguish the third experiment's changed timing and baseline.
- **2:40–3:30 / Honest result:** prior-window 80% versus 60% in one holdout, but one grouped fold loses to baseline. No future or causal claim. Show one false-positive example without identifiers.
- **3:30–4:30 / Recommendation:** open the playbook and local queue. Explain a review reason and mandatory human approval; no automated content action.
- **4:30–5:00 / Reproduce and next step:** show metric receipts and execution commands; propose dated forward validation before operational use.

## ML-12 — Social post draft
My first content-decline model looked strong: 90% Precision@20. Holding out whole clients changed that to 50% with the same features. I audited feature timing, tested a prior-window alternative, and built a human-reviewed action playbook on FlyRank's 30,000-row starter sample. The lesson is methodological: compare against a rule on the same rows, disclose split sensitivity, and keep editorial actions with people. These are retrospective results, not promised traffic gains. Code and measured results: https://github.com/hicham-halfi/Intern-Flyrank

## ML-12 — Employer-facing summary
I built a reproducible audit and editorial review playbook using FlyRank's 30,000-page, 32-client anonymized starter dataset. I reproduced a shallow random forest, tested grouped validation and feature-window restrictions, and documented how an apparent 90% top-20 result fell to 50% after client separation with the same features. The output is a transparent, human-approved review queue with metric receipts, error analysis, monitoring proposals and explicit limits, rather than automated publishing or a causal SEO claim.

## Completion record
Notebook execution and Git commit are verified separately. The deployed URL is recorded only after the public page is observed live. A real-phone check and the assignment's requested personal research/review steps must not be inferred from a desktop preview.
